# Vector stores and semantic search



In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

## Part I: Basic vector store implementation

In [ ]:
class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document


class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self._model = embedding_model
        self._documents: list[Document] = []
        self._embeddings: np.ndarray | None = None

    def add_documents(self, documents: list[Document]):
        texts = [doc.text for doc in documents]
        new_embeddings = self._model.encode(texts, convert_to_numpy=True, show_progress_bar=True)
        # Normalizamos
        norms = np.linalg.norm(new_embeddings, axis=1, keepdims=True)
        new_embeddings = new_embeddings / np.maximum(norms, 1e-10)

        self._documents.extend(documents)
        if self._embeddings is None:
            self._embeddings = new_embeddings
        else:
            self._embeddings = np.vstack([self._embeddings, new_embeddings])

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        if self._embeddings is None or len(self._documents) == 0:
            return []

        query_emb = self._model.encode([query], convert_to_numpy=True)
        query_emb = query_emb / np.maximum(np.linalg.norm(query_emb, keepdims=True), 1e-10)

        scores = (self._embeddings @ query_emb.T).flatten()

        top_indices = np.argsort(scores)[::-1][:top_k]
        return [
            SearchResult(score=float(scores[i]), document=self._documents[i])
            for i in top_indices
        ]

## Part II: Filtering by metadata

In [ ]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        pass

    def add_documents(self, documents: list[Document]):
        pass

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        pass